**Entregable 1** — Consolidación de Datos con Python

1.1 Leer el archivo y consolidar

In [63]:
# Importación de liberias Python
from google.colab import drive
import pandas as pd

# Acceso al Drive
drive.mount('/content/drive')

# Obtenemos la ruta del archivo Service Centers y asignamos al DF
ruta = "/content/drive/MyDrive/Service_centers.csv"
df = pd.read_csv(ruta)

# Imprimimos la info (Columnas, Tipo de dato) del Data Frame
# df.info()

# Casteamos la fecha de apertura
df["fecha_estimada_apertura"] = (
  pd.to_datetime(df["fecha_estimada_apertura"], errors="coerce")
  .dt.strftime("%Y-%m-%d")
)

# Calcula el promedio de las columnas 'obra_civil','compras','licencias' de forma horizontal
df["avance_total"] = df[["avance_obra_civil_pct","avance_compras_pct","avance_licencias_pct"]].mean(axis=1)

# Calcula los dias de retraso tomando la fecha actual
df["dias_retraso"] = (pd.to_datetime("today") - pd.to_datetime(df["fecha_estimada_apertura"])).dt.days

# Seteamos a 0 todos los valores negativos en dias_retraso
df["dias_retraso"] = df["dias_retraso"].clip(lower=0)

# Seleccionamos la columnas a trabajar y sumarizamos dias_retraso si aplica
df_resumen = df.groupby(["id_sc", "estado", "tipo_sc", "avance_obra_civil_pct",
                         "avance_compras_pct", "avance_licencias_pct", "fecha_estimada_apertura",
                         "avance_total"])["dias_retraso"].sum().reset_index()

# print(df_resumen)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1.2 Exportamos los datos a un excel con Formato (no pude usar gspread por restriccion de APIS)

In [64]:
# Importamos las librerias para formatear
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Funcion para formatear el excel
def exportar_excel_formateado(df, ruta_drive):

    wb = Workbook()
    ws = wb.active
    ws.title = "Resumen"

    # ── Estilos ───────────────────────────────────────────────
    color_header     = "434343"
    color_fila_par   = "FEF5D9"
    color_fila_impar = "FEF0C7"

    estilo_borde = Border(
        left=Side(style="thin"),
        right=Side(style="thin"),
        top=Side(style="thin"),
        bottom=Side(style="thin")
    )

    # ── Encabezados ───────────────────────────────────────────
    for col_idx, col_nombre in enumerate(df.columns, start=1):
        celda = ws.cell(row=1, column=col_idx, value=col_nombre)
        celda.font      = Font(bold=True, color="FFFFFF", name="Arial", size=11)
        celda.fill      = PatternFill("solid", start_color=color_header)
        celda.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        celda.border    = estilo_borde

    ws.row_dimensions[1].height = 30

    # ── Datos ─────────────────────────────────────────────────
    for row_idx, row in enumerate(df.itertuples(index=False), start=2):
        color_fila = color_fila_par if row_idx % 2 == 0 else color_fila_impar
        for col_idx, valor in enumerate(row, start=1):
            celda = ws.cell(row=row_idx, column=col_idx, value=valor)
            celda.font      = Font(name="Arial", size=10)
            celda.fill      = PatternFill("solid", start_color=color_fila)
            celda.alignment = Alignment(horizontal="center", vertical="center")
            celda.border    = estilo_borde

            # Formato de 2 decimales para columnas numéricas
            if pd.api.types.is_numeric_dtype(df.iloc[:, col_idx - 1]):
                celda.number_format = "#,##0.00"

    # ── Ancho de columnas automático ──────────────────────────
    for col_idx, col_nombre in enumerate(df.columns, start=1):
        max_ancho = max(
            len(str(col_nombre)),
            df.iloc[:, col_idx - 1].astype(str).map(len).max()
        )
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_ancho + 4, 40)

    wb.save(ruta_drive)
    print(f"✅ Archivo guardado en: {ruta_drive}")


# Guadado del archivo en Drive
ruta = "/content/drive/MyDrive/Service_Centers_ETL.xlsx"
exportar_excel_formateado(df_resumen, ruta)

✅ Archivo guardado en: /content/drive/MyDrive/Service_Centers_ETL.xlsx


**Entregable 2** — Tablero en Looker Studio

https://datastudio.google.com/u/0/reporting/2a866f6b-e9c8-42d4-bd43-63c66c0ebd59/page/FkmwF


**Entregable 3** — Módulo de IA para Detección de Riesgos

In [59]:
import pandas as pd
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('Api_GPT'))

# Convierte el DataFrame a JSON
df_json = df_resumen.to_json(orient="records", force_ascii=False, indent=2)

# Arma el mensaje incluyendo el JSON
prompt = "Eres analista de operaciones logísticas. Analiza el siguiente reporte de aperturas de Service Centers y genera: 1. Un párrafo de resumen ejecutivo (máximo 80 palabras). 2. Los 3 principales riesgos operativos esta semana. 3. Una acción recomendada por cada riesgo."

respuesta = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "Eres un analista de datos. Recibirás datos en formato JSON y responderás preguntas sobre ellos."
        },
        {
            "role": "user",
            "content": f"{prompt}\n\nDatos:\n{df_json}"
        }
    ]
)

print(respuesta.choices[0].message.content)


1. **Resumen Ejecutivo**: Esta semana, el reporte de aperturas de Service Centers revela avances desiguales en los proyectos. Los centros SC-005 en Querétaro y SC-003 en Jalisco están cerca de completarse pero enfrentan retrasos significativos, mientras que SC-012 en Guanajuato y SC-011 en Aguascalientes presentan los menores avances. En general, el retraso de las licencias destaca como un problema común. Es fundamental abordar estos desafíos para asegurar plazos de apertura adecuados.

2. **Principales Riesgos Operativos**:
   
   - **Retrasos en la obtención de licencias**: Es un problema recurrente, especialmente en SC-002 y SC-009, indicando un riesgo de postergar las aperturas.
   
   - **Progreso lento en obra civil y compras**: Proyectos como SC-004 en Nuevo León y SC-011 en Aguascalientes muestran avances lentos, afectando el cronograma general.
   
   - **Acumulación de retrasos en proyectos avanzados**: Los centros SC-005 y SC-010, aunque con alto avance total, presentan retr

**4.- Github**

In [85]:
!git pull origin main --rebase
!git add -f "Worksample_SetUp.ipynb"
!git status  # verifica que aparece en "Changes to be committed"
!git commit -m "feat: actualiza notebook con resolución IA y tablero Looker"
!git push origin main
!git rebase --abort
!git pull origin main --no-rebase
!git push origin main

From https://github.com/alonsohernandez-spec/Worksample_SetUp
 * branch            main       -> FETCH_HEAD
Already up to date.
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	drive/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	drive/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)
Everything up-to-date
fatal: No rebase in progress?
From https://github.com/alonsohernandez-spec/Worksample_SetUp
 * branch            main       -> FETCH_HEAD
Already up to date.
Everything up-to-date
